In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("MovieRatingsAnalytics").getOrCreate()

In [0]:
ratings_data = [
    (1,101,4,1622388000000),
    (1,102,3,1622388020000),
    (2,101,5,1622388040000),
    (2,103,4,1622388060000),
    (3,101,3,1622388080000),
    (3,102,4,1622388100000),
    (3,103,None,1622388120000),
    (4,101,2,1622388140000)
]

ratings_columns = ["user_id","movie_id","rating","timestamp"]

ratings_df = spark.createDataFrame(ratings_data, ratings_columns)

ratings_df.show()

+-------+--------+------+-------------+
|user_id|movie_id|rating|    timestamp|
+-------+--------+------+-------------+
|      1|     101|     4|1622388000000|
|      1|     102|     3|1622388020000|
|      2|     101|     5|1622388040000|
|      2|     103|     4|1622388060000|
|      3|     101|     3|1622388080000|
|      3|     102|     4|1622388100000|
|      3|     103|  NULL|1622388120000|
|      4|     101|     2|1622388140000|
+-------+--------+------+-------------+



In [0]:
movies_data = [
    (101,"The Last Kingdom","Action",2018),
    (102,"Love in Paris","Romance",2020),
    (103,"Galaxy Warriors","Sci-Fi",2019),
    (104,"Mystery Mansion","Thriller",2021),
    (105,"Laugh Out Loud","Comedy",2017),
    (106,"The Silent Forest","Drama",2016),
    (107,"Fast Track","Action",2022),
    (108,"Deep Ocean Secrets","Documentary",2015)
]

movies_columns = ["movie_id","movie_name","genre","release_year"]

movies_df = spark.createDataFrame(movies_data, movies_columns)

movies_df.show()

+--------+------------------+-----------+------------+
|movie_id|        movie_name|      genre|release_year|
+--------+------------------+-----------+------------+
|     101|  The Last Kingdom|     Action|        2018|
|     102|     Love in Paris|    Romance|        2020|
|     103|   Galaxy Warriors|     Sci-Fi|        2019|
|     104|   Mystery Mansion|   Thriller|        2021|
|     105|    Laugh Out Loud|     Comedy|        2017|
|     106| The Silent Forest|      Drama|        2016|
|     107|        Fast Track|     Action|        2022|
|     108|Deep Ocean Secrets|Documentary|        2015|
+--------+------------------+-----------+------------+



In [0]:
clean_df = ratings_df.dropna()

clean_df.show()

+-------+--------+------+-------------+
|user_id|movie_id|rating|    timestamp|
+-------+--------+------+-------------+
|      1|     101|     4|1622388000000|
|      1|     102|     3|1622388020000|
|      2|     101|     5|1622388040000|
|      2|     103|     4|1622388060000|
|      3|     101|     3|1622388080000|
|      3|     102|     4|1622388100000|
|      4|     101|     2|1622388140000|
+-------+--------+------+-------------+



In [0]:
clean_df = clean_df.dropDuplicates()

clean_df.show()

+-------+--------+------+-------------+
|user_id|movie_id|rating|    timestamp|
+-------+--------+------+-------------+
|      1|     101|     4|1622388000000|
|      1|     102|     3|1622388020000|
|      2|     101|     5|1622388040000|
|      2|     103|     4|1622388060000|
|      3|     101|     3|1622388080000|
|      3|     102|     4|1622388100000|
|      4|     101|     2|1622388140000|
+-------+--------+------+-------------+



In [0]:
valid_ratings_df = clean_df.filter(
    (col("rating") >= 1) &
    (col("rating") <= 5)
)

valid_ratings_df.show()

+-------+--------+------+-------------+
|user_id|movie_id|rating|    timestamp|
+-------+--------+------+-------------+
|      1|     101|     4|1622388000000|
|      1|     102|     3|1622388020000|
|      2|     101|     5|1622388040000|
|      2|     103|     4|1622388060000|
|      3|     101|     3|1622388080000|
|      3|     102|     4|1622388100000|
|      4|     101|     2|1622388140000|
+-------+--------+------+-------------+



In [0]:
movie_ratings_df = valid_ratings_df.join(
    movies_df,
    on="movie_id",
    how="inner"
)

movie_ratings_df.show()

+--------+-------+------+-------------+----------------+-------+------------+
|movie_id|user_id|rating|    timestamp|      movie_name|  genre|release_year|
+--------+-------+------+-------------+----------------+-------+------------+
|     101|      1|     4|1622388000000|The Last Kingdom| Action|        2018|
|     102|      1|     3|1622388020000|   Love in Paris|Romance|        2020|
|     101|      2|     5|1622388040000|The Last Kingdom| Action|        2018|
|     103|      2|     4|1622388060000| Galaxy Warriors| Sci-Fi|        2019|
|     101|      3|     3|1622388080000|The Last Kingdom| Action|        2018|
|     102|      3|     4|1622388100000|   Love in Paris|Romance|        2020|
|     101|      4|     2|1622388140000|The Last Kingdom| Action|        2018|
+--------+-------+------+-------------+----------------+-------+------------+



In [0]:
average_ratings_df = movie_ratings_df.groupBy(
    "movie_name"
).agg(
    round(avg("rating"),2).alias("average_rating"),
    count("rating").alias("total_ratings")
)

average_ratings_df.show()

+----------------+--------------+-------------+
|      movie_name|average_rating|total_ratings|
+----------------+--------------+-------------+
|The Last Kingdom|           3.5|            4|
|   Love in Paris|           3.5|            2|
| Galaxy Warriors|           4.0|            1|
+----------------+--------------+-------------+



In [0]:
trending_movies_df = average_ratings_df.orderBy(
    desc("average_rating"),
    desc("total_ratings")
)

trending_movies_df.show()

+----------------+--------------+-------------+
|      movie_name|average_rating|total_ratings|
+----------------+--------------+-------------+
| Galaxy Warriors|           4.0|            1|
|The Last Kingdom|           3.5|            4|
|   Love in Paris|           3.5|            2|
+----------------+--------------+-------------+



In [0]:
user_activity_df = movie_ratings_df.groupBy(
    "user_id"
).agg(
    count("movie_id").alias("movies_rated"),
    round(avg("rating"),2).alias("avg_user_rating")
)

user_activity_df.show()

+-------+------------+---------------+
|user_id|movies_rated|avg_user_rating|
+-------+------------+---------------+
|      1|           2|            3.5|
|      2|           2|            4.5|
|      3|           2|            3.5|
|      4|           1|            2.0|
+-------+------------+---------------+



In [0]:
valid_ratings_df.write.mode("overwrite").saveAsTable(
    "clean_ratings_table"
)
